## Notebook Overview

This notebook provides a **full experimental environment** for Noise2Noise denoising of XRF data: loading detector frames, preparing the dataset, defining hyperparameter sweeps, training multiple models, performing tiled inference, storing and organizing results, and interactively examining how each training configuration affects the denoised output. It is designed as a practical toolbox for **systematic benchmarking and model selection** in XRF denoising workflows.

### Main steps

1. **Setup and data loading**  
   - Mount Google Drive, enable mixed precision, and load the `setkafluo` library.  
   - Select an element (e.g. *Zn-K*) and load all corresponding detector-specific TIFF frames.  
   - Explore the stack interactively to inspect detector variability and noise levels.

2. **Dataset preparation**  
   - Exclude selected detector indices and convert the remaining frames into a clean `(N, H, W)` array.  
   - Load the weighted-sum evaluation image (`fluo1`) used for inference and visual comparison.

3. **Benchmark configuration**  
   - Define baseline hyperparameters (patch size, batch size, steps per epoch, epochs).  
   - Specify single-parameter sweeps:  
     - **PATCH:** 8 → 32  
     - **BATCH:** 4 → 32  
     - **STEPS:** 16 → 64  
   - Automatically create training folders, prediction folders, and a manifest file to log all runs.

4. **Training multiple models (parameter sweeps)**  
   - For each sweep condition, build a U-Net with the appropriate configuration.  
   - Train the model on the detector-frame stack using Noise2Noise pairing and patch extraction.  
   - Save final model weights and record all metadata in the manifest.

5. **Tiled inference and prediction saving**  
   - Rebuild each model from its saved weights.  
   - Run **tiled prediction** on the weighted-sum evaluation image (`fluo1`) using overlap blending.  
   - Save the resulting denoised TIFF into the benchmarking directory and update the manifest.

6. **Interactive comparison of denoised images**  
   - Browse all runs stored in the manifest.  
   - Select two models at a time and view their predictions side-by-side.  
   - Optional features:  
     - joint or independent contrast,  
     - show/hide original image,  
     - instant refresh.

7. **Pretrained models for Zn line (100 epochs)**  
   - Discover long-trained models automatically by scanning the directory.  
   - Run inference for all discovered configurations.  
   - Use an identical interactive viewer to compare predictions from these deeper models.

---

## Copyright & License

**© 2025 Dmitry Karpov**  
Assistant Professor of Physics and Materials Science, Université Grenoble Alpes  
(Materials Modelling and Exploration Laboratory, MEM – UGA/CEA)

These notebooks were created as **educational material** for the  
**DIADEM Academy training series on U-Net and image analysis** (PEPR DIADEM)

Unless otherwise stated, all notebook content is distributed under the:

**Creative Commons Attribution–NonCommercial 4.0 International License (CC BY-NC 4.0)**  
https://creativecommons.org/licenses/by-nc/4.0/

You are free to **use, share, adapt, and modify** this material for **non-commercial research and teaching**,  
provided that **proper attribution** is given to the author.

Commercial use is **not permitted** without prior written permission.

### Please cite when using or adapting this material:
D. Karpov, *U-Net for Image Analysis — DIADEM Academy Training Materials*, Université Grenoble Alpes, 2025-present.

### Disclaimer
This material is provided *“as is”*, without warranty of any kind.  
The author and associated institutions are not liable for any damages arising from its use.

---

In [ ]:
# this line is just to check GPU status
!nvidia-smi

In [ ]:
# this cell is needed to mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, re
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import Button, IntSlider, HBox, VBox, HTML, Layout, Output, Dropdown, Checkbox
from IPython.display import display
from mpl_toolkits.axes_grid1 import make_axes_locatable

import tensorflow as tf
from tensorflow.keras import mixed_precision



# mixed precision for speed on Colab GPUs
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

In [ ]:
# set the path to the setkafluo library in the directory that
# you cloned with the notebooks
# Be careful! Sometimes Colab needs change of My Drive to MyDrive or vice versa
sys.path.append("/content/drive/MyDrive/setkafluo_demo/notebooks_and_library/libs/")

import setkafluo as skf
print("SetkaFluo version:", skf.__version__)

from setkafluo.denoise import (
    extract_covering_patches_with_overlap_pad,
    generate_noise2noise_samples,
    standardize_images,
    extract_random_patches,
    augment_patch,
    make_unet, predict_tiled, save_tiff
)

In [ ]:
# The path to the base folder
base_path = Path("/content/drive/MyDrive/setkafluo_demo/")

training_path = base_path / "training/" # your path where the training will happen
data_dir  = base_path / "input_data/" # path to the data

# We will define ckpt_dir / pred_dir right after
# selecting `element` in the next cell
def make_output_dirs(element: str):
    month = datetime.now().strftime("%b")
    ckpt_dir = training_path / f"training/{element}/{month}"
    pred_dir = training_path / f"training/{element}/predictions"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    pred_dir.mkdir(parents=True, exist_ok=True)
    return ckpt_dir, pred_dir

In [ ]:
# Pick the element you want to work with (e.g. "K-K", "Zn-K", "Fe-K", …)
# Zn-K is probably the most visual example !
element = "Zn-K"

# Create output directories for this element
ckpt_dir, pred_dir = make_output_dirs(element)

def read_tif_files(directory_path, ardesia=True, element='Zn-K', elements_to_discard=None):
    """
    Returns:
        idxs (np.ndarray of int)
        images (list[np.ndarray])  <-- keep as list to avoid object-dtype arrays
        names (np.ndarray of str)
    """
    suffix = "_ardesia" if ardesia else "_arwen"
    records = []  # (idx, image, filename)

    for fname in os.listdir(directory_path):
        if fname.endswith(f"_{element}_area_density_ngmm2.tif") and "_element" in fname and suffix in fname:
            i0 = fname.index("_element") + len("_element")
            i1 = fname.index("_", i0)
            idx = int(fname[i0:i1])
            if elements_to_discard and idx in elements_to_discard:
                continue
            img = skf.load_tiff(os.path.join(directory_path, fname))
            records.append((idx, img, fname))

    # sort by index
    records.sort(key=lambda t: t[0])

    if not records:
        return np.array([], dtype=int), [], np.array([], dtype=str)

    idxs  = np.array([r[0] for r in records], dtype=int)
    imgs  = [r[1] for r in records]                 # list of np.ndarrays
    names = np.array([r[2] for r in records], dtype=str)
    return idxs, imgs, names


In [ ]:
# Interactive visualization: click/slide through the stack with indices and filenames shown
data_dir_stack = data_dir / "processed/separate_elements/"

# Load the full stack (no exclusions yet)
idxs, images_stack, names = read_tif_files(str(data_dir_stack), ardesia=True, element=element)
print(f"Loaded {len(images_stack)} frames for {element}. Indexes: {idxs}")

if len(images_stack) == 0:
    print("No images found for the selected element.")
else:
    def to_numeric_img(x):
        arr = np.asarray(x)
        if arr.dtype == object:
            arr = arr.astype(np.float32)
        return arr

    def auto_contrast(img):
        p1, p99 = np.percentile(img, [1, 99])
        if p1 == p99:
            p1, p99 = float(np.min(img)), float(np.max(img))
        return p1, p99

    # Widgets
    slider = IntSlider(value=0, min=0, max=len(images_stack)-1, step=1,
                       description='Frame:', continuous_update=False)
    btn_prev = Button(description="⟵ Prev", layout=Layout(width="100px"))
    btn_next = Button(description="Next ⟶", layout=Layout(width="100px"))
    info = HTML()
    controls = HBox([btn_prev, slider, btn_next])
    ui = VBox([controls, info])

    out = Output()

    def draw(k: int):
        img = to_numeric_img(images_stack[k])
        vmin, vmax = auto_contrast(img)

        out.clear_output(wait=True)
        with out:
            # Larger figure so the image is bigger
            fig, ax = plt.subplots(figsize=(8.5, 8.5))
            im = ax.imshow(img, vmin=vmin, vmax=vmax)  # default colormap
            ax.set_title(f"{element}  •  detector index = {idxs[k]}", fontsize=12)
            ax.set_axis_off()

            # Colorbar with same height as the image axis
            divider = make_axes_locatable(ax)
            cax = divider.append_axes("right", size="3%", pad=0.05)
            cb = fig.colorbar(im, cax=cax)

            # Ensure colorbar height matches the image height exactly
            ax.set_box_aspect(img.shape[0] / img.shape[1])

            fig.tight_layout()
            plt.show()

        info.value = (
            f"<code>{k+1}/{len(images_stack)}</code> | "
            f"idx=<b>{idxs[k]}</b> | "
            f"min={np.min(img):.3g}, mean={np.mean(img):.3g}, max={np.max(img):.3g}"
        )

    def on_prev(_):
        if slider.value > slider.min:
            slider.value -= 1
            draw(slider.value)

    def on_next(_):
        if slider.value < slider.max:
            slider.value += 1
            draw(slider.value)

    def on_slide(change):
        if change["name"] == "value":
            draw(change["new"])

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    slider.observe(on_slide)

    display(ui, out)
    draw(slider.value)

In [ ]:
# After browsing above, set the indexes you want to exclude
bad_elements = [15]

# Reload the stack without the bad indexes
idxs, images_stack, names = read_tif_files(
    str(data_dir_stack), ardesia=True, element=element, elements_to_discard=bad_elements
)

print("Indexes:", idxs)
print("Files:", names)

# Ensure we have a proper NumPy stack for training: (N, H, W)
if isinstance(images_stack, list):
    try:
        images_stack = np.stack(images_stack, axis=0)  # will fail if shapes differ
    except ValueError as e:
        # If this happens, your images are different sizes.
        # You can either resize/pad them or drop the offenders.
        raise ValueError(
            f"Could not stack images into a single array because shapes differ. "
            f"Please inspect and fix the inconsistent images. Original error: {e}"
        )

print("Stack shape:", images_stack.shape, images_stack.dtype)

In [ ]:
# Evaluation (weighted sum) image
data_dir_weight = data_dir / "processed/weighted_sums/"

fluo1 = skf.load_tiff(
    data_dir_weight / f"IMG_fluo1_{element}_area_density_ngmm2.tif"
)
print("Eval image:", fluo1.shape, fluo1.dtype)

# Visualize with matched colorbar height
p1, p99 = np.percentile(fluo1, [1, 99])
if p1 == p99:
    p1, p99 = float(np.min(fluo1)), float(np.max(fluo1))

fig, ax = plt.subplots(figsize=(7.5, 7.5))
im = ax.imshow(fluo1, vmin=p1, vmax=p99)  # default colormap
ax.set_title(f"Weighted sum {element}")
ax.set_axis_off()

# Make colorbar axis with same height as image axis
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.05)
cb = fig.colorbar(im, cax=cax)

# Preserve image aspect so the colorbar height matches
ax.set_box_aspect(fluo1.shape[0] / fluo1.shape[1])

fig.tight_layout()
plt.show()

# Benchmark Setup (Defaults, Sweeps, Folders)

**Goal:** Define default hyperparameters and **single-parameter sweeps**; prepare directories for a small parameter study.

**What this cell does**
- Sets **defaults** (used unless overridden):
  - `PATCH_DEFAULT = 32`
  - `BATCH_DEFAULT = 4`
  - `EPOCHS_DEFAULT = 10` (short for demo)
  - `STEPS_DEFAULT = 64`
- Defines **grids** (vary **one** knob at a time):
  - `PATCH_GRID = [8, 32]`
  - `BATCH_GRID = [4, 32]`
  - `STEPS_GRID = [16, 64]`
- Creates the benchmarking folder layout:
  - `.../benchmarking/models` — saved final weights.
  - `.../benchmarking/predicted` — prediction TIFFs.
  - `.../benchmarking/manifest.json` — run registry.

**Quick reference**

| Knob   | Meaning                               | Default | Grid         |
|:------:|----------------------------------------|:-------:|:------------:|
| PATCH  | Tile size (px)                         |   32    |   8, 32      |
| BATCH  | Batch size (patch pairs per step)      |    4    |   4, 32      |
| STEPS  | Steps per epoch                        |   64    |  16, 64      |
| EPOCHS | Epochs                                  |   10    | (fixed)      |

> **Note:** When sweeping one parameter, the other two remain at their defaults.

In [ ]:
# ---- Benchmark setup (defaults + sweeps + folders) ----
# ---- Defaults ----
PATCH_DEFAULT  = 32
BATCH_DEFAULT  = 4
EPOCHS_DEFAULT = 10      # fixed for the demo
STEPS_DEFAULT  = 64

# ---- Sweeps (vary ONE knob at a time; others stay default) ----
PATCH_GRID = [8, 32]
BATCH_GRID = [4, 32]
STEPS_GRID = [16, 64]

# ---- Output dirs ----
BENCH_DIR       = training_path / f"training/{element}/benchmarking/"
BENCH_MODELS    = BENCH_DIR / "models"
BENCH_PRED_DIR  = BENCH_DIR / "predicted"

for d in (BENCH_DIR, BENCH_MODELS, BENCH_PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Baseline config builder ----
def make_base_cfg():
    return skf.DenoiseConfig(
        lr=1e-5,
        batch_size=BATCH_DEFAULT,
        patch_size=PATCH_DEFAULT,
        steps_per_epoch=STEPS_DEFAULT,
        epochs=EPOCHS_DEFAULT,
        # model
        input_channels=1, output_channels=1,
        start_ch=64, depth=4, inc_rate=2.0, dropout=0.0,
        instancenorm=False, averagepool=False, upconv=False, residual=False,
        lambda_reg=0.0, reg_l1=False,
        # inference
        min_overlap=PATCH_DEFAULT // 2,
        # checkpoints: disable periodic saving; we'll save final only
        save_models_dir=None, save_every_epochs=999999,
        mixed_precision_warn=True,
    )

# ---- Helper to create a run record name & dict (kept in a manifest) ----
def run_tag(patch, batch, steps, epochs):
    return f"patch{patch}_batch{batch}_steps{steps}_epochs{epochs}"

def run_cfg(overrides: dict):
    cfg = make_base_cfg()
    if "patch_size" in overrides:
        cfg.patch_size = int(overrides["patch_size"])
        cfg.min_overlap = max(0, cfg.patch_size // 2)
    if "batch_size" in overrides:
        cfg.batch_size = int(overrides["batch_size"])
    if "steps_per_epoch" in overrides:
        cfg.steps_per_epoch = int(overrides["steps_per_epoch"])
    if "epochs" in overrides:
        cfg.epochs = int(overrides["epochs"])
    return cfg

# A manifest we keep updating across cells (path set here for consistency)
MANIFEST_PATH = BENCH_DIR / "manifest.json"
print("Benchmark dirs:")
print("  models   ->", BENCH_MODELS)
print("  predicted->", BENCH_PRED_DIR)
print("  manifest ->", MANIFEST_PATH)

# Train sweeps (vary one knob at a time; save final model weights)
This step takes about 10 minutes on T4 machine

In [ ]:
manifest = {
    "element": element,
    "created": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "runs": []
}

# ---- Helper to train one run and save final weights ----
def train_and_save(overrides: dict, sweep: str):
    cfg = run_cfg(overrides)
    tag = run_tag(cfg.patch_size, cfg.batch_size, cfg.steps_per_epoch, cfg.epochs)
    weights_path = BENCH_MODELS / f"{sweep}__{tag}.weights.h5"
    print(f"\n[train] sweep={sweep} tag={tag}")
    print("  -> patch:", cfg.patch_size, "batch:", cfg.batch_size, "steps:", cfg.steps_per_epoch, "epochs:", cfg.epochs)
    model, history = skf.train(images_stack, cfg)   # periodic ckpt is disabled via cfg
    model.save_weights(str(weights_path))
    tf.keras.backend.clear_session()
    print("  saved final weights:", weights_path)

    # record run
    manifest["runs"].append({
        "sweep": sweep,
        "tag": tag,
        "params": {
            "patch_size": cfg.patch_size,
            "batch_size": cfg.batch_size,
            "steps_per_epoch": cfg.steps_per_epoch,
            "epochs": cfg.epochs
        },
        "weights_path": str(weights_path),
        "pred_path": None  # will be filled in the inference cell
    })

# ---- PATCH sweep (BATCH & STEPS stay default) ----
for P in PATCH_GRID:
    train_and_save({"patch_size": P, "batch_size": BATCH_DEFAULT, "steps_per_epoch": STEPS_DEFAULT, "epochs": EPOCHS_DEFAULT}, sweep="PATCH")

# ---- BATCH sweep (PATCH & STEPS stay default) ----
for B in BATCH_GRID:
    train_and_save({"patch_size": PATCH_DEFAULT, "batch_size": B, "steps_per_epoch": STEPS_DEFAULT, "epochs": EPOCHS_DEFAULT}, sweep="BATCH")

# ---- STEPS sweep (PATCH & BATCH stay default) ----
for S in STEPS_GRID:
    train_and_save({"patch_size": PATCH_DEFAULT, "batch_size": BATCH_DEFAULT, "steps_per_epoch": S, "epochs": EPOCHS_DEFAULT}, sweep="STEPS")

# Save/overwrite manifest
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)
print("\nWrote manifest:", MANIFEST_PATH)

# Inference for all runs -> TIFFs in BENCH_PRED_DIR

In [ ]:
# We'll use the 'fluo1' image already loaded earlier in your notebook
assert 'fluo1' in globals(), "Expected eval image 'fluo1' to be defined earlier."

with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

for r in manifest["runs"]:
    params = r["params"]
    cfg = run_cfg(params)  # build cfg consistent with training (incl. min_overlap)
    tag = r["tag"]
    weights_path = r["weights_path"]

    print(f"\n[infer] {tag}")

    # Rebuild model with the same cfg and load final weights
    model = make_unet(cfg)          # no compile needed for inference
    model.load_weights(weights_path)

    # Predict (tiled) and save
    pred = skf.predict_tiled(fluo1, model, cfg)
    pred_path = BENCH_PRED_DIR / f"{tag}.tif"
    skf.save_tiff(pred, pred_path)
    print("  saved:", pred_path)

    # Record path in manifest
    r["pred_path"] = str(pred_path)

    # Free graph memory
    del model
    tf.keras.backend.clear_session()

# Persist updated manifest
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print("\nUpdated manifest with prediction paths.")

# Interactive viewer: compare predicted TIFFs by parameters

In [ ]:
with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

# Load predicted runs
pred_runs = [
    r for r in manifest["runs"]
    if r.get("pred_path") and os.path.exists(r["pred_path"])
]

# Sort runs deterministically
pred_sorted = sorted(
    pred_runs,
    key=lambda r: (
        r["sweep"],
        r["params"]["patch_size"],
        r["params"]["batch_size"],
        r["params"]["steps_per_epoch"],
    ),
)

def _label(r):
    p = r["params"]
    return f"{r['sweep']}: PATCH={p['patch_size']}  BATCH={p['batch_size']}  STEPS={p['steps_per_epoch']}"

# Dropdown options
OPTIONS = [("Original image", "ORIGINAL")] + [
    (_label(r), i) for i, r in enumerate(pred_sorted)
]

# Widgets
dd_a = Dropdown(options=OPTIONS, value=0, description="A:", layout=Layout(width="450px"))
dd_b = Dropdown(options=OPTIONS, value=1 if len(OPTIONS) > 1 else 0,
                description="B:", layout=Layout(width="450px"))

joint_contrast = Checkbox(value=True, description="Joint contrast", indent=False)
refresh_btn = Button(description="Refresh", layout=Layout(width="100px"))

info = HTML()
out = Output()

def _pcts(a, p=(1,99)):
    p1, p99 = np.percentile(a, list(p))
    if p1 == p99:
        p1, p99 = float(np.min(a)), float(np.max(a))
    return p1, p99

def _load_or_original(choice):
    if choice == "ORIGINAL":
        return fluo1.astype(np.float32), "Original image"
    else:
        r = pred_sorted[choice]
        return skf.load_tiff(r["pred_path"]).astype(np.float32), _label(r)

def draw():
    out.clear_output(wait=True)

    # Load A and B
    imgA, labelA = _load_or_original(dd_a.value)
    imgB, labelB = _load_or_original(dd_b.value)

    # Determine contrast
    if joint_contrast.value:
        lo = min(_pcts(imgA)[0], _pcts(imgB)[0])
        hi = max(_pcts(imgA)[1], _pcts(imgB)[1])
        vA = vB = (lo, hi)
    else:
        vA = _pcts(imgA)
        vB = _pcts(imgB)

    with out:
        fig, axs = plt.subplots(1, 2, figsize=(13, 6))
        axs = np.atleast_1d(axs)

        # Left panel
        im = axs[0].imshow(imgA, vmin=vA[0], vmax=vA[1])
        axs[0].set_title(f"A — {labelA}")
        axs[0].axis("off")
        cax = make_axes_locatable(axs[0]).append_axes("right", size="3%", pad=0.05)
        fig.colorbar(im, cax=cax)
        axs[0].set_box_aspect(imgA.shape[0] / imgA.shape[1])

        # Right panel
        im = axs[1].imshow(imgB, vmin=vB[0], vmax=vB[1])
        axs[1].set_title(f"B — {labelB}")
        axs[1].axis("off")
        cax = make_axes_locatable(axs[1]).append_axes("right", size="3%", pad=0.05)
        fig.colorbar(im, cax=cax)
        axs[1].set_box_aspect(imgB.shape[0] / imgB.shape[1])

        fig.tight_layout()
        plt.show()

    info.value = (
        f"<b>A:</b> {labelA}<br>"
        f"<b>B:</b> {labelB}"
    )

def _on_change(_):
    draw()

dd_a.observe(_on_change, names="value")
dd_b.observe(_on_change, names="value")
joint_contrast.observe(_on_change, names="value")
refresh_btn.on_click(lambda _: draw())

controls = VBox([
    HBox([dd_a, dd_b], layout=Layout(gap="15px")),
    HBox([joint_contrast, refresh_btn], layout=Layout(gap="10px")),
])

display(VBox([controls, info, out]))
draw()

# Pretrained models with more epochs
Here you will explore the images inferred from the models that have been pretrained for 100 epochs. By default only Zn model is included but you can train other elements for 100 epochs if needed

In [ ]:
# ---- Defaults ----
PATCH_DEFAULT  = 32
BATCH_DEFAULT  = 4
EPOCHS_DEFAULT = 100      # long by default (for pre-trained runs)
STEPS_DEFAULT  = 64

# ---- Sweeps (vary ONE knob at a time; others stay default) ----
PATCH_GRID = [8, 32]
BATCH_GRID = [4, 32]
STEPS_GRID = [16, 64]

# ---- Output dirs ----
BENCH_DIR       = training_path / f"{element}/benchmarking_prepared/" # path to the models with pretraining
BENCH_MODELS    = training_path / f"{element}/benchmarking_prepared/models"
BENCH_PRED_DIR  = training_path / f"{element}/benchmarking_prepared/predicted"

for d in (BENCH_DIR, BENCH_MODELS, BENCH_PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Baseline config builder ----
def make_base_cfg():
    return skf.DenoiseConfig(
        lr=1e-5,
        batch_size=BATCH_DEFAULT,
        patch_size=PATCH_DEFAULT,
        steps_per_epoch=STEPS_DEFAULT,
        epochs=EPOCHS_DEFAULT,
        # model
        input_channels=1, output_channels=1,
        start_ch=64, depth=4, inc_rate=2.0, dropout=0.0,
        instancenorm=False, averagepool=False, upconv=False, residual=False,
        lambda_reg=0.0, reg_l1=False,
        # inference
        min_overlap=PATCH_DEFAULT // 2,
        # checkpoints: disable periodic saving; we'll save final only
        save_models_dir=None, save_every_epochs=999999,
        mixed_precision_warn=True,
    )

def run_cfg(overrides: dict):
    """Clone baseline & apply overrides, keeping min_overlap in sync with patch."""
    cfg = make_base_cfg()
    if "patch_size" in overrides:
        cfg.patch_size = int(overrides["patch_size"])
        cfg.min_overlap = max(0, cfg.patch_size // 2)
    if "batch_size" in overrides:
        cfg.batch_size = int(overrides["batch_size"])
    if "steps_per_epoch" in overrides:
        cfg.steps_per_epoch = int(overrides["steps_per_epoch"])
    if "epochs" in overrides:
        cfg.epochs = int(overrides["epochs"])
    return cfg

def run_tag(patch, batch, steps, epochs):
    return f"patch{patch}_batch{batch}_steps{steps}_epochs{epochs}"

# ---- Manifest path ----
MANIFEST_PATH = BENCH_PRED_DIR / "manifest.json"

# ---- Auto-discovery: build a manifest by scanning BENCH_MODELS if manifest is missing ----
_tag_re = re.compile(
    r"""(?:.+__)?patch(?P<patch>\d+)_batch(?P<batch>\d+)_steps(?P<steps>\d+)_epochs(?P<epochs>\d+)\.weights\.h5$""",
    re.IGNORECASE
)

def _scan_models_to_manifest():
    runs = []
    for f in sorted(BENCH_MODELS.glob("*.weights.h5")):
        m = _tag_re.search(f.name)
        if not m:
            continue
        p = {k: int(m.group(k)) for k in ("patch", "batch", "steps", "epochs")}
        tag = run_tag(p["patch"], p["batch"], p["steps"], p["epochs"])
        runs.append({
            "sweep": "AUTO",
            "tag": tag,
            "params": {
                "patch_size": p["patch"],
                "batch_size": p["batch"],
                "steps_per_epoch": p["steps"],
                "epochs": p["epochs"],
            },
            "weights_path": str(f),
            "pred_path": None
        })
    manifest = {
        "element": element,
        "created": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "runs": runs
    }
    return manifest

def ensure_manifest():
    """Load manifest if present; otherwise auto-create from weights in BENCH_MODELS."""
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH, "r") as f:
            return json.load(f)
    manifest = _scan_models_to_manifest()
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"[manifest] Created from models scan -> {MANIFEST_PATH}")
    return manifest

print("Benchmark dirs:")
print("  models   ->", BENCH_MODELS)
print("  predicted->", BENCH_PRED_DIR)
print("  manifest ->", MANIFEST_PATH)

### This cell will run for quite a long time in case you want to do the 100 epochs training using T4 machine. So keep it commented in case you want to see Zn denoising for which the models are included in the datasets on Zenodo.

### If you want to train other elements then you will need to change the previosly selected element and uncomment this cell

In [ ]:
# # LONG TRAINING !!! DO NOT UNCOMMENT !!!
# # LONG TRAINING !!! DO NOT UNCOMMENT !!!
# # LONG TRAINING !!! DO NOT UNCOMMENT !!!

# # ---- Train sweeps (vary one knob at a time; save final model weights) ----
# manifest = {
#     "element": element,
#     "created": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
#     "runs": []
# }

# def train_and_save(overrides: dict, sweep: str):
#     cfg = run_cfg(overrides)
#     tag = run_tag(cfg.patch_size, cfg.batch_size, cfg.steps_per_epoch, cfg.epochs)
#     weights_path = BENCH_MODELS / f"{sweep}__{tag}.weights.h5"
#     print(f"\n[train] sweep={sweep} tag={tag}")
#     print("  -> patch:", cfg.patch_size, "batch:", cfg.batch_size, "steps:", cfg.steps_per_epoch, "epochs:", cfg.epochs)

#     model, history = skf.train(images_stack, cfg)   # periodic ckpt disabled via make_base_cfg
#     model.save_weights(str(weights_path))
#     tf.keras.backend.clear_session()
#     print("  saved final weights:", weights_path)

#     manifest["runs"].append({
#         "sweep": sweep,
#         "tag": tag,
#         "params": {
#             "patch_size": cfg.patch_size,
#             "batch_size": cfg.batch_size,
#             "steps_per_epoch": cfg.steps_per_epoch,
#             "epochs": cfg.epochs
#         },
#         "weights_path": str(weights_path),
#         "pred_path": None
#     })

# # ---- PATCH sweep (BATCH & STEPS stay default) ----
# for P in PATCH_GRID:
#     train_and_save({"patch_size": P, "batch_size": BATCH_DEFAULT, "steps_per_epoch": STEPS_DEFAULT, "epochs": EPOCHS_DEFAULT}, sweep="PATCH")

# # ---- BATCH sweep (PATCH & STEPS stay default) ----
# for B in BATCH_GRID:
#     train_and_save({"patch_size": PATCH_DEFAULT, "batch_size": B, "steps_per_epoch": STEPS_DEFAULT, "epochs": EPOCHS_DEFAULT}, sweep="BATCH")

# # ---- STEPS sweep (PATCH & BATCH stay default) ----
# for S in STEPS_GRID:
#     train_and_save({"patch_size": PATCH_DEFAULT, "batch_size": BATCH_DEFAULT, "steps_per_epoch": S, "epochs": EPOCHS_DEFAULT}, sweep="STEPS")

# # Save/overwrite manifest
# with open(MANIFEST_PATH, "w") as f:
#     json.dump(manifest, f, indent=2)
# print("\nWrote manifest:", MANIFEST_PATH)

In [ ]:
# ---- Inference for all runs -> TIFFs in BENCH_PRED_DIR ----
# Ensure we have a manifest (load or auto-create by scanning models)
manifest = ensure_manifest()

# Make sure eval image exists
assert 'fluo1' in globals(), "Expected eval image 'fluo1' to be defined earlier."

updated = False
for r in manifest["runs"]:
    params = r["params"]
    cfg = run_cfg(params)                      # match training cfg for tiling/min_overlap
    tag = r["tag"]
    weights_path = r["weights_path"]
    pred_path = BENCH_PRED_DIR / f"{tag}.tif"

    # Skip if prediction already exists on disk & recorded
    if pred_path.exists() and r.get("pred_path") == str(pred_path):
        print(f"[infer] {tag} -> already exists, skipping")
        continue

    print(f"\n[infer] {tag}")
    model = make_unet(cfg)                     # no compile needed for predict()
    model.load_weights(weights_path)

    pred = predict_tiled(fluo1, model, cfg)    # from submodule import
    save_tiff(pred, pred_path)
    print("  saved:", pred_path)

    r["pred_path"] = str(pred_path)
    updated = True

    # Free graph memory
    del model
    tf.keras.backend.clear_session()

# Persist updated manifest if changed
if updated:
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)
    print("\nUpdated manifest with prediction paths.")
else:
    print("\nManifest already up to date.")

In [ ]:
# ---- Interactive viewer: compare predicted TIFFs by parameters ----

with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

# Collect predicted runs
runs = [
    r for r in manifest["runs"]
    if r.get("pred_path") and os.path.exists(r["pred_path"])
]

if not runs:
    print("No predictions found in manifest. Run the inference cell first (or check BENCH_PRED_DIR).")
else:
    # Sort consistently
    runs_sorted = sorted(
        runs,
        key=lambda r: (
            r.get("sweep", ""),
            r["params"]["patch_size"],
            r["params"]["batch_size"],
            r["params"]["steps_per_epoch"]
        )
    )

    def _label(r):
        p = r["params"]
        return (
            f"{r.get('sweep','?')}: "
            f"PATCH={p['patch_size']}  "
            f"BATCH={p['batch_size']}  "
            f"STEPS={p['steps_per_epoch']}"
        )

    # Dropdown options: Original + predicted runs
    OPTIONS = [("Original image", "ORIGINAL")] + [
        (_label(r), i) for i, r in enumerate(runs_sorted)
    ]

    # Widgets
    dd_a = Dropdown(options=OPTIONS, value=0,
                    description="A:", layout=Layout(width="450px"))
    dd_b = Dropdown(options=OPTIONS, value=1 if len(OPTIONS) > 1 else 0,
                    description="B:", layout=Layout(width="450px"))

    joint_contrast = Checkbox(value=True, description="Joint contrast", indent=False)
    refresh_btn = Button(description="Refresh", layout=Layout(width="100px"))

    info = HTML()
    out = Output()

    def _pcts(a, p=(1, 99)):
        p1, p99 = np.percentile(a, list(p))
        if p1 == p99:
            p1, p99 = float(np.min(a)), float(np.max(a))
        return p1, p99

    def _load_or_original(choice):
        if choice == "ORIGINAL":
            return fluo1.astype(np.float32), "Original image"
        else:
            r = runs_sorted[choice]
            return skf.load_tiff(r["pred_path"]).astype(np.float32), _label(r)

    def draw():
        out.clear_output(wait=True)

        # Load image A
        imgA, labelA = _load_or_original(dd_a.value)
        # Load image B
        imgB, labelB = _load_or_original(dd_b.value)

        # Determine contrast
        if joint_contrast.value:
            lo = min(_pcts(imgA)[0], _pcts(imgB)[0])
            hi = max(_pcts(imgA)[1], _pcts(imgB)[1])
            vA = vB = (lo, hi)
        else:
            vA = _pcts(imgA)
            vB = _pcts(imgB)

        with out:
            fig, axs = plt.subplots(1, 2, figsize=(13, 6))
            axs = np.atleast_1d(axs)

            # Panel A
            im = axs[0].imshow(imgA, vmin=vA[0], vmax=vA[1])
            axs[0].set_title(f"A — {labelA}")
            axs[0].axis("off")
            cax = make_axes_locatable(axs[0]).append_axes("right", size="3%", pad=0.05)
            fig.colorbar(im, cax=cax)
            axs[0].set_box_aspect(imgA.shape[0] / imgA.shape[1])

            # Panel B
            im = axs[1].imshow(imgB, vmin=vB[0], vmax=vB[1])
            axs[1].set_title(f"B — {labelB}")
            axs[1].axis("off")
            cax = make_axes_locatable(axs[1]).append_axes("right", size="3%", pad=0.05)
            fig.colorbar(im, cax=cax)
            axs[1].set_box_aspect(imgB.shape[0] / imgB.shape[1])

            fig.tight_layout()
            plt.show()

        info.value = (
            f"<b>A:</b> {labelA}<br>"
            f"<b>B:</b> {labelB}"
        )

    def _on_change(_):
        draw()

    dd_a.observe(_on_change, names="value")
    dd_b.observe(_on_change, names="value")
    joint_contrast.observe(_on_change, names="value")
    refresh_btn.on_click(lambda _: draw())

    controls = VBox([
        HBox([dd_a, dd_b], layout=Layout(gap="15px")),
        HBox([joint_contrast, refresh_btn], layout=Layout(gap="10px")),
    ])

    display(VBox([controls, info, out]))
    draw()